In [ ]:
!pip install datasets

In [ ]:
!pip install transformers datasets torch scikit-learn -q

In [ ]:
from datasets import load_dataset
print("Datasets installed successfully")

Datasets installed successfully


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,confusion_matrix
from transformers import AutoTokenizer,AutoModelForSequenceClassification, Trainer,TrainingArguments, IntervalStrategy

In [ ]:
print("Gpu Avalilable:",torch.cuda.is_available())

Gpu Avalilable: True


In [ ]:
df = pd.read_csv('/content/IMDB_dataset.csv', engine='python', on_bad_lines='skip')
df.sample(500, random_state=42)

,review,sentiment
17902,This film is so much of a rip-off of the maste...,negative
3916,"Yes, I admire the independent spirit of it all...",negative
21582,"Put a DVD of this flick in a time capsule, and...",negative
3050,"Once again, I've been duped by seemingly intel...",negative
18566,"As low budget indies go, you will usually find...",negative
...,...,...
1921,This movie is painfully slow and has no plot. ...,negative
13474,"This game requires stealth, smart, and a stead...",positive
19408,If you are studying Welles and want to see jus...,negative
3711,here was no effort put into Valentine to preve...,negative


In [ ]:
df['sentiment']=df['sentiment'].map({'positive':1, 'negative':0})
df=df.dropna()
df=df.rename(columns={"review":"text","sentiment":"label"})

In [ ]:
train_texts, temp_texts,train_labels, temp_labels=train_test_split(df['text'],df['label'],test_size=0.3,random_state=42)
val_texts,test_texts,val_labels,test_labels=train_test_split(
    temp_texts,temp_labels,test_size=0.5,random_state=42)



In [ ]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings=tokenizer(list(train_texts),truncation=True,padding=True,max_length=128)
val_encodings=tokenizer(list(val_texts),truncation=True,padding=True,max_length=128)
test_encodings=tokenizer(list(test_texts),truncation=True,padding=True,max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
class Dataset(torch.utils.data.Dataset):
  def __init__(self,encodings,labels):
     self.encodings=encodings
     self.labels=list(labels)

  def __getitem__(self,idx):
         item={key:torch.tensor(val[idx]) for key, val in self.encodings.items()}
         item['labels']=torch.tensor(self.labels[idx])
         return item


  def __len__(self):
      return len(self.labels)

In [ ]:
train_dataset=Dataset(train_encodings,train_labels)
val_dataset=Dataset(val_encodings,val_labels)
test_dataset=Dataset(test_encodings,test_labels)

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import IntervalStrategy

training_args=TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir='./logs',
    save_strategy="no"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
def compute_metrics(pred):
  labels=pred.label_ids
  preds=pred.predictions.argmax(-1)

  precision,recall, f1, _ =precision_recall_fscore_support(labels,preds,average="binary")
  acc=accuracy_score(labels,preds)

  return{
      'accuracy':acc,
      'f1':f1,
      'precision':precision,
      'recall':recall
  }


In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.298212
1000,0.280117
1500,0.269247
2000,0.306331
2500,0.338318


TrainOutput(global_step=2634, training_loss=0.2985794136898936, metrics={'train_runtime': 295.4591, 'train_samples_per_second': 71.303, 'train_steps_per_second': 8.915, 'total_flos': 697672671871488.0, 'train_loss': 0.2985794136898936, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.309395432472229,
 'eval_accuracy': 0.8785999113867966,
 'eval_f1': 0.8804015713662157,
 'eval_precision': 0.8727823453050627,
 'eval_recall': 0.8881549977983267,
 'eval_runtime': 18.7931,
 'eval_samples_per_second': 240.194,
 'eval_steps_per_second': 30.064,
 'epoch': 1.0}

In [ ]:
preds=trainer.predict(test_dataset)
y_pred=preds.predictions.argmax(-1)

cm=confusion_matrix(test_labels,y_pred)
print(cm)

[[1939  314]
 [ 261 2001]]
